# TP2 - Etapa 2: entrenamiento, evaluación y comparación de modelos

Notebook de entrenamiento y estudio de resultados: evaluación de la búsqueda por similitud
(Etapa 1, NDCG@10), fine-tuning de ResNet (Modelo A), CNN propia (Modelo B) y comparación entre
modelos (ver `ENUNCIADO.md`, secciones 4.3 y 5). **Ambos modelos son obligatorios.**

El entrenamiento se realiza en Colab para aprovechar la GPU:
**Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU (T4)**.

Requisitos previos (en tu fork):
- Funciones de la Etapa 1 implementadas en `src/lib/services/similarity_service.py`.
- `train_classifier`, `evaluate_classifier` y `extract_custom_embedding` implementadas en
  `src/lib/services/classifier_service.py` (esta notebook solo las orquesta).
- Splits y conjunto independiente generados en `1-pre-processing.ipynb`.

Al finalizar:
- Descargar los checkpoints generados y colocarlos en `models/` del entorno local
  (la aplicación los usa en las pestañas de las Etapas 1, 2 y 3).
- Publicarlos en un link público de solo lectura para los docentes.
- Entregar esta notebook ejecutada, con sus salidas.

**Notebook en Google Colab (si corresponde):** <reemplazar por el link, con permiso de lectura para los docentes>

Si entregan la notebook en Colab, esta copia del repositorio funciona como notebook de referencia:
el link de arriba es obligatorio y la notebook de Colab debe estar ejecutada, con sus salidas
(ver `ENUNCIADO.md`, sección 8.2).

## Equipo
- Alumno 1 :
- Alumno 2 :

## 1. Entorno

La notebook corre en Colab (GPU T4) o localmente desde la raíz del repositorio. En Colab:

- Se monta Google Drive y ahí se guardan **checkpoints, historiales y resultados de experimentos**
  (`MyDrive/tp2-perros/`): si Colab se desconecta no se pierde nada, y al volver a ejecutar las celdas
  los experimentos ya terminados no se repiten.
- El dataset se copia al disco local de la máquina (`/content`), no se lee desde Drive: leer miles de
  JPG chicos desde Drive hace que la GPU espere al disco.

Si el fork es privado, usar un token de acceso en `REPO_URL` (no dejarlo escrito en la notebook entregada).

In [ ]:
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/lucasramire/tuia-dog-recognition-app.git"
BRANCH = "feature/valentin-lujambio"  # pasar a main cuando este integrado

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    if not Path("/content/proyecto").exists():
        !git clone --branch $BRANCH $REPO_URL /content/proyecto
    %cd /content/proyecto
    !git log --oneline -1
    !pip install -q pydantic-settings python-dotenv kagglehub albumentations onnx onnxruntime
print("Colab:", IN_COLAB)

## 2. Configuración

Las rutas se fijan como variables de entorno **antes** de importar `lib` (igual que en
`1-pre-processing.ipynb`). Sin pgvector: la evaluación usa matrices de numpy.

In [ ]:
import json
import logging
import os
import shutil
import time
from dataclasses import asdict

ROOT = Path.cwd()
assert (ROOT / "src" / "lib").is_dir(), "Ejecutar la notebook desde la raiz del repositorio"
PERSIST = Path("/content/drive/MyDrive/tp2-perros") if IN_COLAB else ROOT
os.environ.update(
    {
        "USE_PGVECTOR": "false",
        "DATA_PATH": str(ROOT / "data"),
        "DATASET_PATH": str(ROOT / "data" / "dataset"),
        "EMBEDDINGS_PATH": str(ROOT / "data" / "embeddings.json"),
        "OUTPUT_PATH": str(PERSIST / "output"),
        "MODEL_PATH": str(PERSIST / "models"),
        "TORCH_HOME": str(ROOT / "models" / "torch"),
        "YOLO_MODEL": str(ROOT / "models" / "yolov8n.pt"),
    }
)
sys.path.insert(0, str(ROOT / "src"))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display

from lib.config import settings

logging.getLogger().setLevel(logging.WARNING)
logging.getLogger("lib.training").setLevel(logging.INFO)  # una linea por epoca
plt.rcParams.update({"figure.dpi": 80, "axes.grid": True, "grid.alpha": 0.3})
pd.set_option("display.max_rows", 80)
pd.set_option("display.width", 140)

DATASET = settings.dataset_path
EXP_DIR = settings.output_path / "experiments"
for d in (settings.model_path, EXP_DIR):
    d.mkdir(parents=True, exist_ok=True)
SEED = 42
NUM_WORKERS = min(4, os.cpu_count() or 1)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("dataset:", DATASET)
print("checkpoints:", settings.model_path, "| experimentos:", EXP_DIR)
print("GPU:", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "no", "| workers:", NUM_WORKERS)

## 3. Dataset y splits

El dataset se descarga con `scripts/download_dataset.py` (kagglehub; si pide credenciales, cargar
`KAGGLE_USERNAME` y `KAGGLE_KEY` en los *secrets* de Colab). En Colab se guarda además un zip en Drive
para no volver a descargarlo.

Los splits son los de `1-pre-processing.ipynb`: `data/splits/splits_v1.csv` (sin fugas entre splits ni
duplicados; `valid` y `test` con 10 imágenes por raza). Todo lo que sigue usa **solo `train` y `valid`
para elegir**; `test` se mira una vez por modelo final.

In [ ]:
ZIP = PERSIST / "dataset.zip"
if not (DATASET / "dogs.csv").exists():
    if IN_COLAB and ZIP.exists():
        shutil.unpack_archive(str(ZIP), str(DATASET))
    else:
        !python scripts/download_dataset.py
        if IN_COLAB:
            shutil.make_archive(str(ZIP.with_suffix("")), "zip", str(DATASET))
print(sorted(p.name for p in DATASET.iterdir() if not p.name.startswith(".")))

In [ ]:
from lib.dataset.manifest import class_names, load_manifest

MANIFEST = settings.data_path / "splits" / "splits_v1.csv"
manifest = load_manifest(MANIFEST, dataset_path=DATASET, check_files=True)
CLASSES = class_names(manifest)
counts = manifest.groupby(["breed", "split"]).size().unstack(fill_value=0)[["train", "valid", "test"]]
display(counts.agg(["sum", "min", "median", "max"]).T)
print(f"{len(CLASSES)} razas | train por raza: {counts['train'].min()} a {counts['train'].max()} "
      f"({counts['train'].max() / counts['train'].min():.1f} veces)")

## 5. Etapa 1 — Evaluación de la búsqueda por similitud (baseline)

TO-DO:
- Galería: imágenes de `train`. Consultas: `test` y el conjunto independiente (nunca indexadas).
- Para cada consulta, recuperar las Top-10 y calcular NDCG@10 con relevancia binaria (misma raza).
- Reportar NDCG@10 medio, precision@10 y accuracy de la raza predicha, por separado para `test` y
  para el conjunto independiente.
- Comparar `cosine` vs `l2`, analizar `TOP_K` y calibrar `SIMILARITY_THRESHOLD` con validación.
- Justificar el resultado con ejemplos de consultas y sus Top-10.

Sugerencia: no hace falta la base de datos. Calculá los embeddings de la galería en una matriz de
`numpy` y buscá con un producto matricial (`EmbeddingStore.append` reescribe el JSON en cada llamada).

In [ ]:
from lib.bootstrap import build_similarity, build_store
from lib.evaluation.metrics import ndcg_at_k

similarity = build_similarity(settings, build_store(settings))

## TO-DO: embeddings de la galería con similarity.extract_embedding, búsqueda de las consultas,
## NDCG@10 (ndcg_at_k con n_relevant = imágenes de la raza en la galería), precision@10 y accuracy;
## comparación cosine vs l2 y calibración del umbral.

## Experimentos: cómo se corren

Cada variante se entrena con `lib.training.trainer.train` (la misma función que usa `train_classifier`)
y guarda su checkpoint e historial en `output/experiments/`. Receta común (plan, D-23): AdamW,
scheduler coseno, label smoothing 0,1, weight decay 1e-4, batch 64, precisión mixta en GPU, early stopping
con paciencia 5 sobre la accuracy de `valid`, semilla 42. Se guarda la **mejor época según `valid`**, no la
última.

La selección (estrategia, hiperparámetros) se hace con `valid`. Una vez elegida la variante, el modelo
final se entrena con `classifier.train_classifier(config)`, que escribe el checkpoint que usa la app.

Presupuesto estimado en una T4: 15 a 25 s por época; unas 5 a 6 horas de época en total como máximo,
en la práctica menos por el early stopping.

In [ ]:
from lib.training.trainer import TrainConfig, train
from lib.visualization.training import plot_confusion, plot_histories


def base_config(**overrides) -> TrainConfig:
    return TrainConfig(image_size=settings.image_size, num_workers=NUM_WORKERS, seed=SEED, **overrides)


def run_experiment(name: str, **overrides) -> dict:
    """Entrena una variante; si ya termino antes (archivo .done.json en Drive), reutiliza el resultado."""
    done = EXP_DIR / f"{name}.done.json"
    if done.exists():
        return json.loads(done.read_text())
    start = time.perf_counter()
    result = train(base_config(**overrides), manifest, DATASET, EXP_DIR / f"{name}.pth", EXP_DIR / f"{name}_history.json")
    history = load_history(name)
    summary = {
        "name": name,
        "best_val_accuracy": result.best_val_accuracy,
        "best_epoch": result.best_epoch,
        "epochs_run": result.epochs_run,
        "minutes": (time.perf_counter() - start) / 60,
        "trainable_parameters": history["trainable_parameters"],
        "overrides": overrides,
    }
    done.write_text(json.dumps(summary, indent=2))
    return summary


def load_history(name: str) -> dict:
    return json.loads((EXP_DIR / f"{name}_history.json").read_text())


def results_table(results: list[dict]) -> pd.DataFrame:
    table = pd.DataFrame(results).set_index("name")
    table["minutes"] = table["minutes"].round(1)
    return table[["best_val_accuracy", "best_epoch", "epochs_run", "minutes", "trainable_parameters"]].round(4)

## 6. Modelo A (obligatorio): fine-tuning de ResNet18

**Arquitectura.** ResNet18 preentrenada en ImageNet (`IMAGENET1K_V1`), con la capa `fc` reemplazada por
una cabeza lineal de 512 a 70 razas. El embedding para la búsqueda por similitud es la salida del
*global average pooling* (512 dimensiones), justo antes de la cabeza.

**Estrategias que se comparan** (transfer learning, plan D-21):

| Variante | Qué se entrena | LR | Idea |
|---|---|---|---|
| A1 feature extraction | solo la cabeza (backbone y sus BatchNorm congelados) | cabeza 1e-3 | ImageNet ya tiene ~118 razas de perro: las features genéricas deberían alcanzar bastante |
| A2 fine-tuning completo | todo | cabeza 1e-3, backbone 1e-4 | adapta también las capas profundas; LR menor en el backbone para no borrar lo preentrenado (olvido catastrófico) |
| A3 mejor de A1/A2 + pesos de clase | igual que la mejor | igual | `train` tiene de 65 a 198 imágenes por raza; pesos inversos a la frecuencia en la loss (D-25) |

En feature extraction los BatchNorm del backbone quedan en modo evaluación: si se dejaran en modo
entrenamiento cambiarían sus estadísticas aunque sus pesos estén congelados.

In [ ]:
A_COMMON = dict(model="resnet18_finetuned")
results_a = [
    run_experiment("A1_feature_extraction", **A_COMMON, strategy="feature_extraction"),
    run_experiment("A2_finetune", **A_COMMON, strategy="finetune"),
]
best_strategy = max(results_a, key=lambda r: r["best_val_accuracy"])["overrides"]["strategy"]
results_a.append(run_experiment(f"A3_{best_strategy}_pesos_clase", **A_COMMON, strategy=best_strategy, class_weights=True))
display(results_table(results_a))

In [ ]:
fig = plot_histories({r["name"]: load_history(r["name"]) for r in results_a}, title="Modelo A: variantes")
plt.show()

**Lectura** (corrida en Colab, T4, semilla 42).

| Variante | Mejor acc en `valid` | Mejor época | Épocas corridas | Minutos |
|---|---|---|---|---|
| A1 feature extraction | 0,9286 | 11 | 16 | 6,1 |
| A2 finetune | 0,9386 | 9 | 14 | 6,2 |
| A3 finetune + pesos de clase | **0,9429** | 7 | 12 | 5,3 |

- **El preentrenamiento hace casi todo el trabajo.** Con el backbone congelado y solo 35.910 parámetros
  entrenables (la cabeza lineal), A1 ya pasa el 0,88 en la primera época y llega a 0,93: las features
  de ImageNet separan bien las razas. Descongelar todo (A2, 11,2 M parámetros) suma un punto.
- **Las diferencias están dentro del ruido.** Con 700 imágenes de `valid` el error estándar de una
  accuracy cercana a 0,93 es de un punto, así que 1 a 1,5 puntos entre variantes no alcanzan para
  decir que una es mejor. Elegimos A3 porque es la mejor en `valid` y no cuesta más, no porque
  sepamos que los pesos de clase ayudan. Tiene sentido que su efecto sea chico: el desbalance de
  `train` es moderado (56 a 185 imágenes por raza) y `valid` está balanceado.
- **Sobreajuste.** En el finetune la accuracy de `train` llega a 0,98 o 0,99 mientras `valid` se
  queda en 0,94: memoriza `train` y por eso el early stopping corta pronto (épocas 12 a 14).
  En A1 la brecha es mucho menor porque la cabeza lineal tiene poca capacidad. La accuracy de
  `train` se mide con augmentation y label smoothing, así que la brecha es solo indicativa.

### Modelo A final

Se reentrena la variante ganadora con `train_classifier`, que guarda `models/resnet18_finetuned.pth` y
su historial (`resnet18_finetuned_history.json`). Recién ahí se evalúa sobre `test`.

In [ ]:
from lib.bootstrap import build_classifier

classifier = build_classifier(settings)


def train_final(results: list[dict], **extra) -> dict:
    """Entrena el modelo final con la mejor variante (una sola vez: queda marcado en Drive)."""
    best = max(results, key=lambda r: r["best_val_accuracy"])
    config = base_config(**(best["overrides"] | extra))
    done = settings.model_path / f"{config.model}.done.json"
    if not done.exists():
        result = classifier.train_classifier(config)
        done.write_text(json.dumps({"from": best["name"], "best_val_accuracy": result.best_val_accuracy,
                                    "best_epoch": result.best_epoch, "config": asdict(config)}, indent=2, default=list))
    classifier.set_active_model(config.model)
    return json.loads(done.read_text())


final_a = train_final(results_a)
final_a

In [ ]:
from lib.evaluation.classification import confusion_matrix, per_class_metrics, top_confused_pairs


def evaluate_final(name: str) -> tuple[dict, dict]:
    classifier.set_active_model(name)
    classifier.load_model(name).to(DEVICE)
    metrics = classifier.evaluate_classifier()  # test, metricas macro
    pred = classifier.predict_split("test")
    np.savez(EXP_DIR / f"test_predictions_{name}.npz", y_true=pred["y_true"], y_pred=pred["y_pred"], probs=pred["probs"])
    return metrics, pred


metrics_a, pred_a = evaluate_final("resnet18_finetuned")
pd.Series(metrics_a).round(4)

### Curvas de entrenamiento y matriz de confusión (Modelo A)

In [ ]:
plot_histories({"resnet18_finetuned": json.loads(classifier.history_path("resnet18_finetuned").read_text())},
               title="Modelo A final")
plt.show()


def confusion_report(pred: dict, title: str) -> None:
    cm = confusion_matrix(pred["y_true"], pred["y_pred"], len(pred["classes"]))
    plot_confusion(cm, pred["classes"], title=title)
    plt.show()
    display(pd.DataFrame(top_confused_pairs(cm, pred["classes"], k=10)))
    recall = pd.Series(per_class_metrics(cm)["recall"], index=pred["classes"]).sort_values()
    print("Razas con menor recall en test (n=10 por raza, descriptivo):", recall.head(8).round(2).to_dict())


confusion_report(pred_a, "Modelo A: matriz de confusion normalizada (test)")

**Lectura.** El modelo final (A3 reentrenado, mejor época 7) da en `test` accuracy 0,9243, precision
0,9333, recall 0,9243, especificidad 0,9989 y F1 0,9230. Queda unos dos puntos por debajo de su
mejor `valid` (0,9429). Es lo esperable: `valid` se usó para elegir la variante y la época, así que
su número es optimista. Por eso el resultado que se informa es el de `test`, medido una sola vez.

Los pares más confundidos en `test` son Boston Terrier y Bulldog (6 errores, casi todos de Bulldog
predicho como Boston Terrier), Cockapoo y Labradoodle (4), y Lhasa con Shih-Tzu y Pastor Alemán con
Malinois (3 cada uno). **Son los mismos pares que el EDA encontró en el espacio del baseline.**
Border Collie y Collie no aparecen entre los diez primeros. Si el baseline sin entrenar y el modelo
ajustado fallan en los mismos pares, el error viene sobre todo del parecido real entre las razas
(cruzas como Cockapoo y Labradoodle, o razas que se distinguen por el pelaje o el tamaño) y de
etiquetas dudosas, más que del modelo. La raza con menor recall es Bulldog (0,3) y le sigue Cockapoo
(0,6). Con 10 imágenes por raza en `test`, el recall por raza es descriptivo: un error mueve 10 puntos.

## 7. Modelo B (obligatorio): CNN propia

**Arquitectura** (`src/lib/models/cnn_custom.py`, plan D-22): 5 bloques conv 3x3, BatchNorm, ReLU y
max-pool 2x2 con 32, 64, 128, 256 y 256 canales; *global average pooling*; capa lineal de embedding
de 256 con ReLU; dropout 0,3; clasificador lineal de 70. Se entrena desde cero (inicialización de Kaiming).

- **Por qué así**: con ~100 imágenes por raza y sin preentrenamiento el riesgo principal es el
  sobreajuste. BatchNorm estabiliza y acelera el entrenamiento; GAP en vez de aplanar reduce mucho los
  parámetros (aplanar 256x7x7 a 256 serían 3,2 M solo en esa capa) y da algo de invariancia a la
  posición; el dropout regulariza la cabeza; la capa de embedding expone la representación para la
  búsqueda por similitud.
- LR 1e-3 con AdamW; hasta 60 épocas en el modelo final (desde cero tarda más en converger que el A).

In [ ]:
from lib.models.cnn_custom import CustomCNN

cnn = CustomCNN(CLASSES)
shapes, x = [], torch.zeros(1, 3, settings.image_size, settings.image_size)
for name, block in [*(("bloque %d" % (i + 1), b) for i, b in enumerate(cnn.features)), ("gap", cnn.pool)]:
    x = block(x)
    shapes.append({"capa": name, "salida": tuple(x.shape[1:]), "parametros": sum(p.numel() for p in block.parameters())})
shapes.append({"capa": "embedding", "salida": (cnn.embedding_dim,), "parametros": sum(p.numel() for p in cnn.embedding.parameters())})
shapes.append({"capa": "clasificador", "salida": (cnn.num_classes,), "parametros": sum(p.numel() for p in cnn.classifier.parameters())})
arch = pd.DataFrame(shapes).set_index("capa")
display(arch)
print(f"Total: {sum(p.numel() for p in cnn.parameters()):,} parametros")

**Tabla de experimentos** (hasta 30 épocas cada uno para entrar en el presupuesto; el final va a 60).
Cada variante cambia **una sola cosa** respecto de la base, para poder atribuirle el efecto:

| Variante | Cambio | Hipótesis |
|---|---|---|
| B1 base | — | referencia |
| B2 dropout 0,5 | más regularización en la cabeza | si B1 sobreajusta (train >> valid), debería ayudar |
| B3 augmentation fuerte | recortes de hasta 25 %, rotación de 20°, más brillo y contraste, más probables | más variedad simula más datos; si es excesiva, la red no llega a aprender (subajuste) |
| B4 estadísticas de `train` | la red re-normaliza la entrada con la media y el desvío de `train` | con BatchNorm después de la primera conv el efecto debería ser chico: la conv absorbe el cambio afín de la entrada |
| B5 sin BatchNorm | quita las capas BatchNorm | entrenamiento más lento e inestable desde cero; muestra para qué está BatchNorm |

**Por qué normalizar la entrada (contexto de B4).** Restar la media y dividir por el desvío de cada canal
deja los canales en escalas comparables (ninguno domina el gradiente) y el problema de optimización queda
mejor condicionado. Con un modelo preentrenado hay que usar las estadísticas con las que se entrenó: el
Modelo A las lee de los pesos de torchvision (`IMAGENET1K_V1.transforms()`) y verifica que coincidan con las
del preprocesamiento compartido; si no, el modelo no se construye. Entrenando desde cero, lo natural es
calcularlas sobre el propio `train`, que es lo que prueba B4 contra la base (ImageNet).

B4 no rompe el pipeline único: la imagen sigue llegando normalizada con ImageNet y la renormalización
vive dentro del modelo (buffers del checkpoint), así la app no cambia.

In [ ]:
B_EPOCHS = 30
STRONG_AUG = dict(crop_border=0.25, crop_p=0.7, rotate_degrees=20.0, rotate_p=0.6, brightness=0.3, contrast=0.3, color_p=0.7)
B_COMMON = dict(model="cnn_custom", epochs=B_EPOCHS)
results_b = [
    run_experiment("B1_base", **B_COMMON),
    run_experiment("B2_dropout_0.5", **B_COMMON, cnn_dropout=0.5),
    run_experiment("B3_aug_fuerte", **B_COMMON, augmentation=STRONG_AUG),
    run_experiment("B4_stats_train", **B_COMMON, cnn_train_stats=True),
    run_experiment("B5_sin_batchnorm", **B_COMMON, cnn_batchnorm=False),
]
display(results_table(results_b))

In [ ]:
fig = plot_histories({r["name"]: load_history(r["name"]) for r in results_b}, title="Modelo B: variantes")
plt.show()

**Lectura.**

| Variante | Mejor acc en `valid` | Mejor época |
|---|---|---|
| B1 base | **0,5571** | 29 |
| B4 estadísticas de `train` | 0,5529 | 28 |
| B5 sin BatchNorm | 0,5443 | 26 |
| B2 dropout 0,5 | 0,4771 | 28 |
| B3 augmentation fuerte | 0,4571 | 27 |

- **Ninguna variante terminó de aprender en 30 épocas.** Todas tienen la mejor época al final, y
  las curvas de `valid` siguen subiendo. La comparación es entonces "con el mismo presupuesto", no
  "en convergencia", y eso condiciona cómo se lee cada hipótesis.
- **B2 y B3 empeoran porque el modelo subajusta, no porque sobreajuste.** La hipótesis de B2 pedía
  que B1 sobreajustara, y eso no pasa: la accuracy de `train` de B1 queda pegada a la de `valid`.
  Más dropout o más augmentation regularizan a un modelo que todavía no aprendió lo suficiente, y
  lo frenan: en B2 y B3 la accuracy de `train` (medida con dropout y augmentation) queda incluso
  por debajo de la de `valid`. Con más épocas la augmentation fuerte podría pagar, pero con este
  presupuesto no.
- **B4 confirma la teoría de la normalización de entrada.** Usar la media y el desvío de `train` en
  vez de los de ImageNet cambia 0,4 puntos, que es ruido: la BatchNorm después de la primera
  convolución ya re-centra y re-escala las activaciones, así que la red se adapta igual a cualquiera
  de las dos normalizaciones. Por eso el pipeline único con estadísticas de ImageNet no le cuesta
  nada al Modelo B.
- **B5 sin BatchNorm queda a un punto de B1.** Esperábamos que sin BatchNorm el entrenamiento fuera
  más lento e inestable. Con cinco bloques, inicialización de Kaiming y AdamW la red entrena igual, y
  la diferencia está dentro del ruido (error estándar de unos 1,9 puntos con accuracy de 0,55). Nos
  quedamos con BatchNorm porque no rinde peor y hace al modelo menos sensible a la escala de la entrada.
- Como era de esperar, el Modelo B queda muy por debajo del A: no tiene preentrenamiento y aprende
  todo de unas 7.200 imágenes.

La variante final es **B1**, reentrenada con un tope de 60 épocas, ya que la diferencia más clara es
que con 30 no le alcanzaba.

### Modelo B final

La mejor variante, con hasta 60 épocas, entrenada con `train_classifier` (`models/cnn_custom.pth`).

In [ ]:
final_b = train_final(results_b, epochs=60)
metrics_b, pred_b = evaluate_final("cnn_custom")
pd.Series(metrics_b).round(4)

### Curvas de entrenamiento y matriz de confusión (Modelo B)

In [ ]:
plot_histories({"cnn_custom": json.loads(classifier.history_path("cnn_custom").read_text())}, title="Modelo B final")
plt.show()
confusion_report(pred_b, "Modelo B: matriz de confusion normalizada (test)")

**Lectura.**

- **Más épocas era lo que faltaba.** Con un tope de 60, B1 llega a 0,7157 en `valid` en la época 54
  (contra 0,5571 con 30), y el early stopping corta en la 59.
- **El modelo sigue sin sobreajustar.** En la mejor época la accuracy de `train` es 0,735 y la de
  `valid` 0,716: está cerca de su límite de capacidad con estos datos, no memorizando.
- **Resultados en `test`:** accuracy 0,7029, precision 0,7213, recall 0,7029, especificidad 0,9957 y
  F1 0,6983. Igual que en el Modelo A, queda algo por debajo de `valid`.
- **Los errores están más repartidos que en el A.** La matriz de confusión tiene mucho más ruido
  fuera de la diagonal.
  - Siguen apareciendo pares del EDA, como Cockapoo con Labradoodle (5 errores, el primero) y Lhasa
    con Shih-Tzu.
  - Aparecen confusiones nuevas entre razas que no se parecen tanto: Cairn y Yorkie, Basenji y Corgi,
    Bull Mastiff y Pug.
  - La lectura es que el Modelo B se apoya en señales más gruesas, como el color, el pelaje o el
    tamaño aparente.
  - Las razas con menor recall son Cocker y Chihuahua (0,2 cada una).

## 8. Evaluación sobre el conjunto independiente

TO-DO: las mismas métricas sobre el conjunto independiente para ambos modelos. Comparar con `test`
y discutir el cambio de dominio.

In [ ]:
## TO-DO: métricas de ambos modelos sobre el conjunto independiente.

## 9. Embeddings de los modelos entrenados

TO-DO:
- NDCG@10 de la Etapa 1 con los embeddings del baseline, del Modelo A y del Modelo B
  (`classifier.extract_custom_embedding`), con la misma galería y las mismas consultas.
- Visualización PCA o t-SNE de los embeddings de `test` de cada modelo, lado a lado y coloreada
  por raza (con un subconjunto de razas).

In [ ]:
## TO-DO: NDCG@10 por modelo de embeddings y visualización PCA / t-SNE.

## 10. Análisis de errores

TO-DO: ejemplos de falsos positivos y falsos negativos con la imagen y su score, razas con peor
desempeño e hipótesis sobre las causas.

In [ ]:
## TO-DO: análisis de errores con ejemplos visuales.

## 11. Costo computacional

TO-DO: cantidad de parámetros, tamaño en MB, tiempo de inferencia por imagen en CPU y dimensión
del embedding de cada modelo.

In [ ]:
## TO-DO: costo computacional de cada modelo.

## 12. Comparación y elección final

### Modelo A contra Modelo B sobre `test`

Las dos predicciones son sobre las **mismas** 700 imágenes, así que se comparan de forma pareada:

- **Intervalo bootstrap del 95 %** de la accuracy de cada modelo (2.000 remuestreos de las 700 imágenes).
- **Prueba de McNemar** (exacta): solo cuentan las imágenes donde un modelo acierta y el otro no. Si
  esos dos conteos son muy distintos, la diferencia no es casualidad. Es más sensible que comparar los
  dos intervalos, porque usa el emparejamiento.

In [ ]:
from lib.evaluation.classification import bootstrap_ci, mcnemar_test

correct_a = pred_a["y_true"] == pred_a["y_pred"]
correct_b = pred_b["y_true"] == pred_b["y_pred"]
rows = []
for name, metrics, correct in [("Modelo A (ResNet18)", metrics_a, correct_a), ("Modelo B (CNN propia)", metrics_b, correct_b)]:
    _, low, high = bootstrap_ci(correct)
    rows.append({"modelo": name, **{k: round(v, 4) for k, v in metrics.items()}, "accuracy IC95": f"[{low:.3f}, {high:.3f}]"})
display(pd.DataFrame(rows).set_index("modelo"))
test = mcnemar_test(correct_a, correct_b)
print(f"McNemar: solo A acierta {test['only_a']}, solo B acierta {test['only_b']}, p = {test['p_value']:.2e}")

**Lectura.**

| Modelo | accuracy en `test` | IC 95 % (bootstrap) | F1 |
|---|---|---|---|
| A (ResNet18 finetune + pesos de clase) | 0,9243 | [0,904; 0,943] | 0,9230 |
| B (CNN propia) | 0,7029 | [0,670; 0,739] | 0,6983 |

- **La diferencia es clara.** Los intervalos no se superponen, y McNemar lo confirma: hay 175
  imágenes que solo acierta A y 20 que solo acierta B (p = 4,3e-32).
- **No es solo que A acierte más: B casi no aporta aciertos propios.** Las 20 imágenes que solo
  acierta B son el 3 % de `test`, así que combinar los dos modelos tendría poco para ganar.
- **Para clasificar, la app usa el Modelo A.**

### Tabla final y elección

TO-DO:
- Tabla comparativa de todas las métricas (Modelo A, Modelo B y baseline en la Etapa 1).
- Qué modelo usa la app para clasificar (Etapas 2 y 3) y qué embeddings para la búsqueda
  (`EMBEDDING_MODEL`), con su justificación.
- Búsqueda por similitud vs. clasificación supervisada: ventajas, limitaciones y cuándo conviene cada una.
- Trade-offs entre precisión, velocidad de inferencia, memoria y complejidad.

In [ ]:
## TO-DO: tabla comparativa final.

## 13. Descargar los checkpoints

Descargalos y colocalos en `models/` de tu entorno local para que la aplicación los use.
Recordá publicarlos en un link público de solo lectura y ponerlo en el README y en el PR.

In [ ]:
!ls -lh "{settings.model_path}"

try:
    from google.colab import files

    for name in (settings.resnet18_model_name, settings.cnn_custom_model_name):
        files.download(str(settings.model_path / name))
        files.download(str(settings.model_path / (Path(name).stem + "_history.json")))
except ImportError:
    print("Fuera de Colab: los checkpoints quedan en", settings.model_path)